# Finalización y control de ejecución del Pipeline

## Propósito

Finalizar la ejecución del pipeline registrando su resultado en la tabla de control `workspace.metadata.ejecucion_control`.

Este notebook determina cuándo una ejecución ha terminado correctamente y establece cuál es la ejecución válida para el corte procesado.

## Flujo

1. Obtener `fecha_inicio`, `fecha_fin` e `id_ejecucion` mediante Task Values desde `00_prepare_pipeline_control`.
2. Validar que los parámetros de ejecución hayan sido recibidos correctamente.
3. Identificar cualquier ejecución anterior marcada como válida para el mismo corte.
4. Desmarcar como válida la ejecución anterior del mismo corte.
5. Actualizar la ejecución actual con:

   * `fecha_fin_ejecucion`
   * `estado = EXITOSA`
   * `es_ejecucion_valida = true`
6. Validar el resultado de la actualización.

## Tabla de control

**Tabla:** `workspace.metadata.ejecucion_control`

La tabla mantiene el historial de las ejecuciones realizadas para cada corte.

## Estados de ejecución

* `EN_PROCESO`: ejecución iniciada pero aún no finalizada.
* `EXITOSA`: ejecución finalizada correctamente.
* `FALLIDA`: ejecución que terminó con error.

## Regla de ejecución válida

Para un mismo corte debe existir únicamente una ejecución marcada como válida.

Cuando un corte es reprocesado y la nueva ejecución termina correctamente:

* La ejecución anterior deja de ser válida.
* La nueva ejecución pasa a ser la ejecución válida.
* Las ejecuciones anteriores permanecen registradas para mantener la trazabilidad.

## Trazabilidad

La tabla permite identificar:

* Qué corte fue procesado.
* Qué ejecución procesó el corte.
* Cuándo inició la ejecución.
* Cuándo terminó.
* Cuál fue su estado.
* Qué ejecución debe ser considerada válida para los procesos posteriores.

## Dependencia

Este notebook depende de `00_prepare_pipeline_control`, que genera el `id_ejecucion`, determina el corte y publica los parámetros mediante Task Values.

## Resultado esperado

Una ejecución finalizada correctamente debe quedar registrada como:

* `estado = EXITOSA`
* `es_ejecucion_valida = true`
* `fecha_fin_ejecucion` con la fecha y hora de finalización.

La ejecución válida será posteriormente utilizada por los procesos Silver y Gold para determinar qué información corresponde procesar cuando existan reprocesos del mismo corte.


In [0]:
# ============================================================
# OBTENCIÓN DE PARÁMETROS DE EJECUCIÓN
# ============================================================

fecha_inicio = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="fecha_inicio",
    debugValue=""
)

fecha_fin = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="fecha_fin",
    debugValue=""
)

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

print(f"Fecha inicio: {fecha_inicio}")
print(f"Fecha fin:     {fecha_fin}")
print(f"ID ejecución:  {id_ejecucion}")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

# ------------------------------------------------------------
# 1. Generar identificador único para el registro de log
# ------------------------------------------------------------

LOG_ID = str(uuid.uuid4())

# ------------------------------------------------------------
# 2. Registrar la ejecución como EN_PROCESO
# ------------------------------------------------------------
# Se registra inmediatamente antes de ejecutar la lógica
# principal del Task.
#
# Esto permite que el Task de error pueda recuperar el
# log_id incluso si este Task falla posteriormente.

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(
    log_id,
    id_ejecucion,
    notebook,
    capa,
    estado,
    fecha_inicio
)
VALUES
(
    '{LOG_ID}',
    '{id_ejecucion}',
    'finalize_pipeline_control',
    'control',
    'EN_PROCESO',
    current_timestamp()
)
""")

# ------------------------------------------------------------
# 3. Publicar log_id como Task Value
# ------------------------------------------------------------
# Permite que log_error_finalize_control pueda recibir
# directamente el identificador del registro creado.

dbutils.jobs.taskValues.set(
    key="log_id",
    value=LOG_ID
)

# ------------------------------------------------------------
# 6. Mostrar información de trazabilidad
# ------------------------------------------------------------

print(f"✓ log_id generado: {LOG_ID}")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# FINALIZACIÓN DE EJECUCIÓN
# ============================================================

TABLA_EJECUCION_CONTROL = "metadata_dataco.supply_chain.hd_ejecucion_control"

delta_control = DeltaTable.forName(
    spark,
    TABLA_EJECUCION_CONTROL
)

# ------------------------------------------------------------
# 1. Desactivar ejecución válida anterior del mismo corte
# ------------------------------------------------------------

(
    delta_control.alias("t")
    .update(
        condition=(
            (F.col("t.fecha_inicio") == F.to_date(F.lit(fecha_inicio))) &
            (F.col("t.fecha_fin") == F.to_date(F.lit(fecha_fin))) &
            (F.col("t.id_ejecucion") != F.lit(id_ejecucion)) &
            (F.col("t.es_ejecucion_valida") == F.lit(True))
        ),
        set={
            "es_ejecucion_valida": "false"
        }
    )
)

# ------------------------------------------------------------
# 2. Marcar ejecución actual como EXITOSA y válida
# ------------------------------------------------------------

(
    delta_control.alias("t")
    .update(
        condition=(
            F.col("t.id_ejecucion") == F.lit(id_ejecucion)
        ),
        set={
            "fecha_fin_ejecucion": "current_timestamp()",
            "estado": "'EXITOSA'",
            "es_ejecucion_valida": "true"
        }
    )
)

print(f"✓ Ejecución finalizada en {TABLA_EJECUCION_CONTROL}: {id_ejecucion}")
print("✓ Estado: EXITOSA")
print("✓ Ejecución marcada como válida.")

In [0]:
# ============================================================
# VALIDACIÓN DE REPROCESOS DEL MISMO CORTE
# ============================================================

display(
    spark.table("metadata_dataco.supply_chain.hd_ejecucion_control")
    .filter(
        (F.col("fecha_inicio") == F.to_date(F.lit(fecha_inicio))) &
        (F.col("fecha_fin") == F.to_date(F.lit(fecha_fin)))
    )
    .orderBy(F.col("fecha_inicio_ejecucion"))
)

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL LOG - EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET
    estado = 'EXITOSO',
    fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")